# TroVE × PBEBench-Lite — RunPod runner

End-to-end notebook to:

1. Check GPU and install dependencies
2. Launch a local vLLM server (with native tool-calling flags)
3. Wait for it to be healthy
4. Run TroVE on PBEBench-Lite with reward-based selection
5. Analyze the JSONL output

## Pod sizing

| Model           | Recommended GPU                | Tensor parallel |
|-----------------|--------------------------------|-----------------|
| `gpt-oss-20b`   | 1× A100 80 GB or 1× H100        | 1               |
| `gpt-oss-120b`  | 2× H100 / A100 80 GB           | 2               |

## Before you start

- Run this notebook from a Jupyter kernel **inside the pod**, with the repo at `/workspace/pbe/symbolic-library-agent` (or wherever you cloned it). Adjust `REPO_ROOT` in the next cell if needed.
- Each cell is idempotent — safe to re-run.
- Cleanup at the bottom kills the vLLM process; if you re-run cells out of order, you may end up with a stale server — use the cleanup cell.

## 1. Configuration

In [ ]:
from pathlib import Path
import os

# Pick the model variant. 20b fits on a single A100/H100; 120b needs TP=2.
MODEL = "openai/gpt-oss-20b"          # or "openai/gpt-oss-120b"
TENSOR_PARALLEL = 1                    # set to 2 for 120b

PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Repo root — change if your clone lives elsewhere on the pod.
REPO_ROOT = Path(os.environ.get("REPO_ROOT", "/workspace/pbe/symbolic-library-agent"))
if not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (REPO_ROOT / "main.py").exists(), f"Could not find main.py under {REPO_ROOT}"
os.chdir(REPO_ROOT)

# Tasks file. Two PBEBench-Lite options ship with the repo:
#   - lite_pilot_tasks.jsonl    : 50-task pilot split (smoke-run default)
#   - lite_tasks_full_og.jsonl  : full Lite split (1008 tasks)
TASKS_FILE = REPO_ROOT / "data/pbebench/lite_pilot_tasks.jsonl"
MAX_PROGRAMS = 5     # PBEBench convention for the lite split

OUT_DIR = REPO_ROOT / "outputs"
OUT_FILE = OUT_DIR / "trove_pbebench_lite_smoke.jsonl"
DEBUG_DIR = REPO_ROOT / "debug_trove_pbebench"
VLLM_LOGS = REPO_ROOT / "vllm_logs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)
VLLM_LOGS.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"MODEL      : {MODEL}  (TP={TENSOR_PARALLEL})")
print(f"BASE_URL   : {BASE_URL}")
print(f"TASKS_FILE : {TASKS_FILE}  (exists={TASKS_FILE.exists()})")
print(f"OUT_FILE   : {OUT_FILE}")

## 2. GPU & dependency check

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv

In [ ]:
# Install repo deps + vLLM. Re-running is a no-op if everything's already there.
!pip install -q -U pip wheel
!pip install -q -r requirements.txt 2>&1 | tail -5
!pip install -q -U "vllm>=0.16.0" 2>&1 | tail -5
import importlib, vllm
print("vllm version:", vllm.__version__)

## 3. Launch vLLM in the background

Required flags for `gpt-oss` native tool calling (vLLM ≥ v0.16.0):

- `--enable-auto-tool-choice`
- `--tool-call-parser openai`
- `--reasoning-parser openai_gptoss`

In [ ]:
import os, subprocess, time, datetime

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = VLLM_LOGS / f"vllm_{PORT}_{ts}.log"
pid_path = VLLM_LOGS / f"vllm_{PORT}_{ts}.pid"

user = os.environ.get("USER", "runpod")
for d in (f"/tmp/{user}-tiktoken-cache", f"/tmp/{user}-tmp"):
    Path(d).mkdir(parents=True, exist_ok=True)
    os.chmod(d, 0o700)
os.environ["TIKTOKEN_CACHE_DIR"] = f"/tmp/{user}-tiktoken-cache"
os.environ["TMPDIR"] = f"/tmp/{user}-tmp"

cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL,
    "--tokenizer", MODEL,
    "--dtype", "auto",
    "--port", str(PORT),
    "--gpu-memory-utilization", "0.95",
    "--tensor-parallel-size", str(TENSOR_PARALLEL),
    "--enable-auto-tool-choice",
    "--tool-call-parser", "openai",
    "--reasoning-parser", "openai_gptoss",
]

log_fh = open(log_path, "w")
vllm_proc = subprocess.Popen(cmd, stdout=log_fh, stderr=subprocess.STDOUT)
pid_path.write_text(str(vllm_proc.pid))
print(f"vLLM started — pid {vllm_proc.pid}")
print(f"log : {log_path}")
print(f"pid : {pid_path}")

In [ ]:
# Wait for the OpenAI-compatible /v1/models endpoint to respond.
# 20b cold-start is ~1–2 min; 120b can be 5–10 min on first launch.
import urllib.request, json, time

READY_TIMEOUT_S = 900   # 15 min
POLL_S = 5

deadline = time.time() + READY_TIMEOUT_S
ready = False
while time.time() < deadline:
    if vllm_proc.poll() is not None:
        print("vLLM exited unexpectedly. Tail of log:")
        print(log_path.read_text()[-4000:])
        raise RuntimeError("vLLM died during startup")
    try:
        with urllib.request.urlopen(f"{BASE_URL}/models", timeout=2) as resp:
            data = json.loads(resp.read())
            print("Ready. /v1/models response:")
            print(json.dumps(data, indent=2)[:600])
            ready = True
            break
    except Exception:
        elapsed = int(READY_TIMEOUT_S - (deadline - time.time()))
        print(f"\rwaiting for vLLM... {elapsed}s elapsed", end="", flush=True)
        time.sleep(POLL_S)

if not ready:
    print("\nTimed out. Tail of log:")
    print(log_path.read_text()[-4000:])
    raise RuntimeError("vLLM never became ready")

## 4. Run TroVE on PBEBench-Lite (smoke run)

Defaults below match the design:

- `--trove-task-family pbebench` — strict `**Solution**` parsing + PBEBench few-shots
- `--trove-selection reward` — reward-based candidate selection (AST tie-break)
- `--trove-k 5` — paper default samples per mode
- `--trove-trim-every 9999` — effectively disable periodic trimming for a 50-task smoke
- `--default-reward pbebench` — PBEBench verifier

In [ ]:
import subprocess, sys

os.environ["VLLM_API_KEY"] = os.environ.get("VLLM_API_KEY", "EMPTY")

cmd = [
    sys.executable, "main.py",
    "--framework",        "trove",
    "--backend",          "vllm",
    "--base-url",         BASE_URL,
    "--model",            MODEL,
    "--trove-task-family", "pbebench",
    "--trove-selection",   "reward",
    "--trove-k",           "5",
    "--trove-trim-every",  "9999",
    "--default-reward",    "pbebench",
    "--max-programs",      str(MAX_PROGRAMS),
    "--tasks-file",        str(TASKS_FILE),
    "--output-file",       str(OUT_FILE),
    "--debug-dir",         str(DEBUG_DIR),
]

print(" ".join(cmd))
print()

# Stream stdout/stderr live.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    rc = proc.wait()
print(f"\nmain.py exited with {rc}")

## 5. Analyze the JSONL output

In [ ]:
!python scripts/analyze_trove_run.py "{OUT_FILE}"

In [ ]:
# Quick peek at one row to confirm telemetry made it through.
import json
with open(OUT_FILE) as f:
    first = json.loads(next(f))
print("keys:", sorted(first.keys()))
for k in ("won_mode", "import_eligible", "tool_call_count", "trove_stopped_reason"):
    print(f"  {k:24s} = {first.get(k)}")
print(f"  library_snapshot size  = {len(first.get('library_snapshot', []))}")

## 6. Cleanup — stop vLLM

Run this when you're done so the GPU is freed for the next experiment.

In [ ]:
import signal, time
if vllm_proc.poll() is None:
    vllm_proc.send_signal(signal.SIGINT)
    try:
        vllm_proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        vllm_proc.kill()
        vllm_proc.wait()
    print("vLLM stopped.")
else:
    print("vLLM was not running.")
log_fh.close()